In [1]:
!pip install trl


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install -U bitsandbytes


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import json
from pathlib import Path
from typing import Any, Dict, Iterable, Optional, Tuple

import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer

# -----------------------
# Load dataset
# -----------------------


W0225 14:50:14.684000 23600 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
def _stringify(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, (list, tuple)):
        return "\n".join(_stringify(item) for item in value if item is not None).strip()
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return str(value).strip()
from transformers.utils import is_bitsandbytes_available


def _extract_input_output(record: Dict[str, Any]) -> Optional[Tuple[str, str]]:
    if "input" in record and "output" in record:
        inp = _stringify(record.get("input"))
        out = _stringify(record.get("output"))
        return inp, out

    instruction = _stringify(record.get("instruction") or record.get("prompt"))
    context = _stringify(record.get("context") or record.get("article") or record.get("text"))
    question = _stringify(record.get("question") or record.get("query"))
    raw_input = _stringify(record.get("input"))

    input_parts = [part for part in [instruction, raw_input, question, context] if part]
    inp = "\n\n".join(input_parts).strip()

    out = _stringify(
        record.get("output")
        or record.get("answer")
        or record.get("response")
        or record.get("completion")
        or record.get("label")
    )

    if not inp or out is None:
        return None
    return inp, out


def _iter_records(raw: Any) -> Iterable[Dict[str, Any]]:
    if isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                yield item
    elif isinstance(raw, dict):
        if "data" in raw and isinstance(raw["data"], list):
            for item in raw["data"]:
                if isinstance(item, dict):
                    yield item
        else:
            yield raw


def load_dataset(path: str) -> list[dict[str, str]]:
    records: list[dict[str, str]] = []
    if path.endswith(".jsonl"):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                record = json.loads(line)
                extracted = _extract_input_output(record)
                if extracted:
                    inp, out = extracted
                    records.append({"input": inp, "output": out})
        return records

    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    for record in _iter_records(raw):
        extracted = _extract_input_output(record)
        if extracted:
            inp, out = extracted
            records.append({"input": inp, "output": out})

    return records


# Fixed dataset path - it's in a subdirectory
dataset_path = "data/medical_meadow_wikidoc/medical_meadow_wikidoc.json"

if not Path(dataset_path).exists():
    raise FileNotFoundError(f"Dataset not found at: {dataset_path}")

print(f"Loading dataset from: {dataset_path}")
training_data = load_dataset(dataset_path)

if not training_data:
    raise ValueError("No training samples found. Check dataset_path and schema.")

print(f"Loaded {len(training_data)} training samples")
print(f"Sample record: {training_data[0]}")

# -----------------------
# Hardware setup
# -----------------------
force_cpu = False
use_gpu = torch.cuda.is_available() and not force_cpu
cuda_bf16 = use_gpu and torch.cuda.is_bf16_supported()

device_map = "auto" if use_gpu else {"": "cpu"}

print(f"\n{'='*50}")
print(f"CUDA available: {use_gpu}")
print(f"GPU: {torch.cuda.get_device_name(0) if use_gpu else 'None (using CPU)'}")
print(f"Using BF16: {cuda_bf16}")
print(f"{'='*50}\n")

dataset_num_proc = 2 if use_gpu else 1
per_device_train_batch_size = 1
gradient_accumulation_steps = 16 if use_gpu else 8
optim = "adamw_8bit" if use_gpu else "adamw_torch"

# -----------------------
# Model & tokenizer
# -----------------------
model_name = "microsoft/Phi-3-mini-4k-instruct"
max_seq_length = 1024

use_4bit = False
bnb_config = None

if use_gpu and is_bitsandbytes_available():
    use_4bit = True
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if cuda_bf16 else torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    print("Using 4-bit quantization")

print(f"Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model: {model_name}")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=(torch.bfloat16 if cuda_bf16 else torch.float16) if use_gpu else torch.float32,
    quantization_config=bnb_config,
    device_map=device_map,
    trust_remote_code=True,
)

# Required for LoRA + 4bit
if use_gpu and use_4bit:
    model = prepare_model_for_kbit_training(model)
    print("Model prepared for k-bit training")

# -----------------------
# Prompt formatting
# -----------------------
def format_prompt(example):
    return (
        f"### Input:\n{example['input']}\n\n"
        f"### Output:\n{example['output']}{tokenizer.eos_token}"
    )

print("Formatting dataset...")
formatted_data = [format_prompt(item) for item in training_data]
dataset = Dataset.from_dict({"text": formatted_data})
print(f"Dataset ready with {len(dataset)} examples")

<>:83: SyntaxWarning: invalid escape sequence '\m'
<>:83: SyntaxWarning: invalid escape sequence '\m'
C:\Users\ALAN THOMAS\AppData\Local\Temp\ipykernel_23600\412927944.py:83: SyntaxWarning: invalid escape sequence '\m'
  dataset_path = "data\medical_meadow_wikidoc.json"


{'input': 'What does "Clear: cell" mean?', 'output': 'Clear cell tumors are part of the surface epithelial-stromal tumor group of Ovarian cancers, accounting for 6% of these neoplastic cases. Clear cell tumors are also associated with the pancreas and salivary glands.\nBenign and borderline variants of this neoplasm are rare, and most cases are malignant.\nTypically, they are cystic neoplasms with polypoid masses that protrude into the cyst.\nOn microscopic pathological examination, they are composed of cells with clear cytoplasm (that contains glycogen) and hob nail cells (from which the glycogen has been secreted).\nThe pattern may be glandular, papillary or solid.'}
CUDA available: True
GPU: NVIDIA GeForce RTX 2050


`torch_dtype` is deprecated! Use `dtype` instead!
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 35,651,584 || all params: 3,856,731,136 || trainable%: 0.9244


In [ ]:
print("Configuring model for training...")
model.config.use_cache = False

# Enable gradient checkpointing for memory efficiency
if use_gpu or True:  # Also helpful on CPU
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled")

# -----------------------
# Training arguments
# -----------------------
print("Setting up training arguments...")
training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=use_gpu and not cuda_bf16,
    bf16=cuda_bf16,
    logging_steps=25,
    optim=optim,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    save_strategy="epoch",
    save_total_limit=2,
    dataloader_pin_memory=use_gpu,
    report_to="none",
    no_cuda=not use_gpu,
)

print("Initializing trainer...")
# Try different SFTTrainer initialization methods for compatibility
trainer = None
try:
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        dataset_num_proc=dataset_num_proc,
        args=training_args,
    )
    print("Trainer initialized with tokenizer parameter")
except TypeError as e:
    print(f"First attempt failed: {e}")
    try:
        trainer = SFTTrainer(
            model=model,
            processing_class=tokenizer,
            train_dataset=dataset,
            dataset_text_field="text",
            max_seq_length=max_seq_length,
            dataset_num_proc=dataset_num_proc,
            args=training_args,
        )
        print("Trainer initialized with processing_class parameter")
    except TypeError as e2:
        print(f"Second attempt failed: {e2}")
        trainer = SFTTrainer(
            model=model,
            train_dataset=dataset,
            args=training_args,
            formatting_func=lambda x: x["text"],
        )
        print("Trainer initialized with formatting_func")

if trainer is None:
    raise RuntimeError("Failed to initialize trainer")

print("\n" + "="*50)
print("Starting training...")
print("="*50 + "\n")

trainer_stats = trainer.train()

print("\n" + "="*50)
print("Training completed!")
print("="*50)

Applying formatting function to train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (6070 > 4096). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

You are not running the flash-attention implementation, expect numerical differences.


Step,Training Loss
25,1.686800
50,1.466600
75,1.454200
100,1.482000
125,1.448900
150,1.422700
175,1.429900
200,1.380500
225,1.363200
250,1.422700


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: f1d8c919-c162-46f0-969d-7741a2c9d901)')' thrown while requesting HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: ded0a514-1bba-4882-b297-3a05bb88b297)')' thrown while requesting HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/conf

In [7]:

# -----------------------
# Save model
# -----------------------
model.save_pretrained("phi3_lora_model")
tokenizer.save_pretrained("phi3_lora_model")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 9ca1ad6d-dbf9-436a-9bad-a8927985e5c4)')' thrown while requesting HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /microsoft/Phi-3-mini-4k-instruct/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 3552b7f2-b2e3-4142-9a5a-f5c62577a400)')' thrown while requesting HEAD https://huggingface.co/microsoft/Phi-3-mini-4k-instruct/resolve/main/conf

('phi3_lora_model\\tokenizer_config.json',
 'phi3_lora_model\\special_tokens_map.json',
 'phi3_lora_model\\chat_template.jinja',
 'phi3_lora_model\\tokenizer.model',
 'phi3_lora_model\\added_tokens.json',
 'phi3_lora_model\\tokenizer.json')